# Coloreo de Grafos con Algoritmos Genéticos

Cuaderno listo para **Google Colab** o **JupyterLab**.

Incluye:
- representación del cromosoma por nodo,
- fitness con penalización por conflictos,
- cruce de un punto,
- mutación por cambio de color,
- gráficos de convergencia,
- visualización del grafo coloreado.


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx


## Datos del problema


In [ ]:
n_nodos = 8
aristas = [
    (0, 1), (0, 2), (0, 3),
    (1, 2), (1, 4),
    (2, 3), (2, 5),
    (3, 5), (3, 6),
    (4, 5), (4, 7),
    (5, 6), (5, 7),
    (6, 7)
]
max_colores = 4

tam_poblacion = 100
n_generaciones = 120
prob_cruce = 0.8
prob_mutacion = 0.08
tam_torneo = 3

random.seed(42)
np.random.seed(42)


## Grafo y representación


In [ ]:
G = nx.Graph()
G.add_nodes_from(range(n_nodos))
G.add_edges_from(aristas)

def crear_individuo():
    return np.random.randint(0, max_colores, size=n_nodos)


## Fitness


In [ ]:
def contar_conflictos(individuo):
    conflictos = 0
    for u, v in aristas:
        if individuo[u] == individuo[v]:
            conflictos += 1
    return conflictos

def cantidad_colores_usados(individuo):
    return len(set(individuo))

def fitness(individuo):
    conflictos = contar_conflictos(individuo)
    colores_usados = cantidad_colores_usados(individuo)
    return -(100 * conflictos + colores_usados)


## Operadores genéticos


In [ ]:
def seleccion_torneo(poblacion):
    candidatos = random.sample(poblacion, tam_torneo)
    return max(candidatos, key=fitness).copy()

def cruce_un_punto(padre1, padre2):
    if random.random() < prob_cruce:
        punto = random.randint(1, n_nodos - 1)
        hijo1 = np.concatenate([padre1[:punto], padre2[punto:]])
        hijo2 = np.concatenate([padre2[:punto], padre1[punto:]])
        return hijo1, hijo2
    return padre1.copy(), padre2.copy()

def mutar(individuo):
    mutado = individuo.copy()
    for i in range(n_nodos):
        if random.random() < prob_mutacion:
            mutado[i] = random.randint(0, max_colores - 1)
    return mutado


## Ejecución del algoritmo genético


In [ ]:
poblacion = [crear_individuo() for _ in range(tam_poblacion)]

mejores_fitness = []
promedios_fitness = []
mejores_conflictos = []
mejores_colores = []

mejor_individuo_global = None
mejor_fit_global = -float('inf')

for generacion in range(n_generaciones):
    nueva_poblacion = []
    while len(nueva_poblacion) < tam_poblacion:
        padre1 = seleccion_torneo(poblacion)
        padre2 = seleccion_torneo(poblacion)
        hijo1, hijo2 = cruce_un_punto(padre1, padre2)
        hijo1 = mutar(hijo1)
        hijo2 = mutar(hijo2)
        nueva_poblacion.append(hijo1)
        if len(nueva_poblacion) < tam_poblacion:
            nueva_poblacion.append(hijo2)
    poblacion = nueva_poblacion
    fitness_poblacion = [fitness(ind) for ind in poblacion]
    mejor_generacion = max(poblacion, key=fitness)
    mejor_fit = fitness(mejor_generacion)
    promedio_fit = np.mean(fitness_poblacion)
    mejores_fitness.append(mejor_fit)
    promedios_fitness.append(promedio_fit)
    mejores_conflictos.append(contar_conflictos(mejor_generacion))
    mejores_colores.append(cantidad_colores_usados(mejor_generacion))
    if mejor_fit > mejor_fit_global:
        mejor_fit_global = mejor_fit
        mejor_individuo_global = mejor_generacion.copy()

conflictos_finales = contar_conflictos(mejor_individuo_global)
colores_finales = cantidad_colores_usados(mejor_individuo_global)

print('Mejor cromosoma encontrado:', mejor_individuo_global.tolist())
print('Conflictos finales:', conflictos_finales)
print('Cantidad de colores usados:', colores_finales)
for nodo, color in enumerate(mejor_individuo_global):
    print(f'Nodo {nodo} -> color {color}')


## Convergencia del fitness


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(mejores_fitness, label='Mejor fitness')
plt.plot(promedios_fitness, label='Fitness promedio')
plt.xlabel('Generación')
plt.ylabel('Fitness')
plt.title('Evolución del algoritmo genético')
plt.legend()
plt.grid(True)
plt.show()


## Conflictos por generación


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(mejores_conflictos, label='Conflictos del mejor individuo')
plt.xlabel('Generación')
plt.ylabel('Cantidad de conflictos')
plt.title('Evolución de conflictos')
plt.legend()
plt.grid(True)
plt.show()


## Grafo coloreado


In [ ]:
plt.figure(figsize=(8, 6))
pos = nx.spring_layout(G, seed=42)
palette = ['red', 'blue', 'green', 'orange', 'purple', 'cyan', 'yellow', 'pink']
node_colors = [palette[c % len(palette)] for c in mejor_individuo_global]

nx.draw(
    G,
    pos,
    with_labels=True,
    node_color=node_colors,
    node_size=900,
    font_size=10,
    edgecolors='black'
)

plt.title('Mejor coloreo encontrado')
plt.show()


## Colores por nodo


In [ ]:
plt.figure(figsize=(8, 5))
plt.bar([f'N{i}' for i in range(n_nodos)], mejor_individuo_global)
plt.xlabel('Nodo')
plt.ylabel('Código de color')
plt.title('Color asignado a cada nodo')
plt.grid(axis='y')
plt.show()


## Ejercicio sugerido

Modificar:
- la cantidad máxima de colores,
- la estructura del grafo,
- la probabilidad de mutación,
- la cantidad de generaciones,

y analizar si cambia la cantidad de conflictos o de colores usados.
